# Topic 9 practice: cleaning and missing values

The tables are built in code, so no CSV is needed. Re-run the setup cell to reset them. Start each question from `df`, `raw`, `pay` or `visits`, work on a copy when you change a column, and give each result a new name.

Tip: a comment line must start with `#`.

In [1]:
import pandas as pd

df = pd.DataFrame({"name":   ["Asha", "Ravi", "Meena", "Kiran", "Dev", "Zoya"],
                   "city":   ["Indore", None, "Pune", "Delhi", None, "Pune"],
                   "age":    [25, 31, None, 28, None, 40],
                   "salary": [50000, 62000, 45000, None, 58000, 71000]})

raw = pd.DataFrame({"item":  ["pen", "book", "bag", "ball"],
                    "price": ["100", "250", "abc", "80"],
                    "qty":   [2, 5, 1, 3],
                    "size":  ["10", "20", "30", "40"]})

pay = pd.DataFrame({"who":    ["Asha", "Ravi", "Meena", "Kiran"],
                    "amount": ["Rs 1,200", "Rs 850", "N/A", "Rs 99"]})

visits = pd.DataFrame({"visitor": ["Asha", "asha ", "RAVI", "ravi", "Meena", "Asha"],
                       "page":    ["home", "home", "cart", "cart", "home", "cart"]})
print(df)
print(raw)
print(pay)
print(visits)

    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    None  31.0  62000.0
2  Meena    Pune   NaN  45000.0
3  Kiran   Delhi  28.0      NaN
4    Dev    None   NaN  58000.0
5   Zoya    Pune  40.0  71000.0
   item price  qty size
0   pen   100    2   10
1  book   250    5   20
2   bag   abc    1   30
3  ball    80    3   40
     who    amount
0   Asha  Rs 1,200
1   Ravi    Rs 850
2  Meena       N/A
3  Kiran     Rs 99
  visitor  page
0    Asha  home
1   asha   home
2    RAVI  cart
3    ravi  cart
4   Meena  home
5    Asha  cart


### Q1 (easy). Count the blanks

In `df`, find how many values are missing in each column, how many are missing in the whole table, and the percent missing in each column (rounded to 1 decimal). Print all three.

In [7]:
# your code here (add a comment saying why for each line)
print(df.isna().sum())
print(df.isna().sum().sum())
print((df.isna().mean()*100).round(1))

name      0
city      2
age       2
salary    1
dtype: int64
5
name       0.0
city      33.3
age       33.3
salary    16.7
dtype: float64


### Q2 (easy). Show the rows

Print the rows of `df` where `age` is missing. Then print the `name` list of the rows that have at least one missing value. Last, print how many rows have no missing value at all.

In [13]:
# your code here (add a comment saying why for each line)
print(df[df["age"].isna()])
print(df[df.isna().any(axis=1)]["name"].tolist())
print(len(df.dropna())) 

    name  city  age   salary
2  Meena  Pune  NaN  45000.0
4    Dev  None  NaN  58000.0
['Ravi', 'Meena', 'Kiran', 'Dev']
2


### Q3 (easy to medium). Drop rows

From `df`, make `q3` by keeping only rows that have at least 3 filled values, with the labels renumbered 0, 1, 2 ... Print `q3` and the names in it. Then print how many rows are left if you drop the rows where `city` is missing.

In [17]:
# your code here (add a comment saying why for each line)
q3 = df.dropna(thresh=3).reset_index(drop=True)
print(q3)
print(q3["name"].tolist())
print(df.dropna(subset=["city"]).count())


    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    None  31.0  62000.0
2  Meena    Pune   NaN  45000.0
3  Kiran   Delhi  28.0      NaN
4   Zoya    Pune  40.0  71000.0
['Asha', 'Ravi', 'Meena', 'Kiran', 'Zoya']
name      4
city      4
age       3
salary    3
dtype: int64


### Q4 (medium). Fill the blanks

On a copy of `df` called `f`, fill `age` with the median of age, `salary` with the mean of salary, and `city` with its most common value. Print `f` and check that no blanks are left (print the total number of blanks).

In [18]:
# your code here (add a comment saying why for each line)
f = df.copy()                                          # copy, because we change columns
f["age"] = f["age"].fillna(f["age"].median())          # median of age = 29.5
f["salary"] = f["salary"].fillna(f["salary"].mean())   # mean of salary = 57200.0
f["city"] = f["city"].fillna(f["city"].mode()[0])      # most common city = Pune
print(f)
print(f.isna().sum().sum())                            # 0 -> no blanks left

    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    Pune  31.0  62000.0
2  Meena    Pune  29.5  45000.0
3  Kiran   Delhi  28.0  57200.0
4    Dev    Pune  29.5  58000.0
5   Zoya    Pune  40.0  71000.0
0


### Q5 (medium). Text that should be numbers

In `raw`, `price` and `size` are stored as text. Make a copy `r5`. Turn `size` into whole numbers. Turn `price` into numbers (bad entries become blank), print how many were bad, fill them with the median price, and make `price` whole numbers. Print `r5.dtypes` and the sum of `size`.

In [19]:
# your code here (add a comment saying why for each line)
r5 = raw.copy()                                               # copy, because we change columns
r5["size"] = r5["size"].astype(int)                           # text -> whole numbers
r5["price"] = pd.to_numeric(r5["price"], errors="coerce")     # bad entry (abc) -> NaN
print(r5["price"].isna().sum())                               # 1 bad entry
r5["price"] = r5["price"].fillna(r5["price"].median())        # median of 100, 250, 80 = 100.0
r5["price"] = r5["price"].astype(int)                         # whole numbers
print(r5.dtypes)
print(r5["size"].sum())                                       # 100

1
item     object
price     int32
qty       int64
size      int32
dtype: object
100


### Q6 (medium to hard). Clean money text

In `pay`, the amount is text like `'Rs 1,200'` and one entry is `'N/A'`. Make a copy `p` and a new column `rupees` that holds whole numbers: remove `'Rs '` and `','`, turn into numbers (bad becomes blank), fill the blank with 0, then whole numbers. Print `p` and the total of `rupees`.

In [20]:
# your code here (add a comment saying why for each line)
p = pay.copy()                                                           # copy, because we add a column
p["rupees"] = p["amount"].str.replace("Rs ", "").str.replace(",", "")    # "Rs 1,200" -> "1200"
p["rupees"] = pd.to_numeric(p["rupees"], errors="coerce")                # "N/A" -> NaN
p["rupees"] = p["rupees"].fillna(0).astype(int)                          # blank -> 0, then whole numbers
print(p)
print(p["rupees"].sum())                                                 # 2149

     who    amount  rupees
0   Asha  Rs 1,200    1200
1   Ravi    Rs 850     850
2  Meena       N/A       0
3  Kiran     Rs 99      99
2149


### Q7 (hard). Duplicates and a count

In `visits`, the same visitor appears with different spelling. Make a copy `v`. Clean `visitor` (remove spaces at the ends, lowercase). Print how many rows are repeats of the same visitor and page. Remove those repeats (keep the first) into `v2`, with labels renumbered. Print `v2`. Last, print how many visits each visitor has in `v2`.

In [21]:
# your code here (add a comment saying why for each line)
v = visits.copy()                                                          # copy, because we change a column
v["visitor"] = v["visitor"].str.strip().str.lower()                        # "asha " and "ASHA" become "asha"
print(v.duplicated(subset=["visitor", "page"]).sum())                      # 2 repeats
v2 = v.drop_duplicates(subset=["visitor", "page"], ignore_index=True)      # keep the first of each, renumber
print(v2)
print(v2.groupby("visitor").size())                                        # visits per visitor

2
  visitor  page
0    asha  home
1    ravi  cart
2   meena  home
3    asha  cart
visitor
asha     2
meena    1
ravi     1
dtype: int64
